# Regression Task on Traffic Dataset (Refactored)

This notebook demonstrates the performance of Gaussian Random Field (GRF) kernels on a traffic speed prediction task, compared against an exact diffusion kernel baseline. 

Refactored to use `gpytorch` and the `grf-gp` library (with local stability subclasses).

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import gpytorch
import networkx as nx
from tqdm.auto import tqdm
from grf_gp.sampler import GRFSampler
from grf_gp.kernels.diffusion import GRFDiffusionKernel
from grf_gp.kernels.general import GRFGeneralKernel
from grf_gp.model import GraphGP

import sys
import os
project_root = os.path.abspath("../../..")
sys.path.append(project_root)

from traffic_utils.preprocessing import load_PEMS
from traffic_utils.plotting import plot_PEMS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)

In [ ]:
# Load and preprocess the PEMS dataset
np.random.seed(1111)
torch.manual_seed(1111)
num_train = 250
MAX_WALK_LENGTH = 3

G, data_train, data_test, data = load_PEMS(num_train=num_train, path=os.path.join(project_root, "data/traffic"))
x_train, y_train = data_train
x_test, y_test = data_test
x, y = data

orig_mean, orig_std = np.mean(y_train), np.std(y_train)
y_train_norm = (y_train - orig_mean) / orig_std
y_test_norm = (y_test - orig_mean) / orig_std

X_train = torch.tensor(x_train).long().to(device)
Y_train = torch.tensor(y_train_norm).to(device).flatten()
X_test = torch.tensor(x_test).long().to(device)
Y_test = torch.tensor(y_test_norm).to(device).flatten()
X_full = torch.tensor(x).long().to(device)

adjacency_matrix = nx.to_numpy_array(G)
adj_tensor = torch.tensor(adjacency_matrix).to(device)

In [ ]:
def train_model(model, likelihood, training_iter=1000, lr=0.01):
    model.train()
    likelihood.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
    
    iterator = tqdm(range(training_iter), desc="Training", leave=False)
    for i in iterator:
        optimizer.zero_grad()
        output = model(model.train_inputs[0])
        loss = -mll(output, model.train_targets)
        loss.backward()
        optimizer.step()
        iterator.set_postfix(loss=loss.item())

def evaluate_model(model, likelihood, x_test, y_test, orig_std):
    model.eval()
    likelihood.eval()
    with torch.no_grad():
        with gpytorch.settings.cholesky_jitter(1e-4):
            mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
            train_output = model(model.train_inputs[0])
            lml = mll(train_output, model.train_targets) * model.train_targets.size(0)
            
            observed_pred = likelihood(model(x_test))
            mean = observed_pred.mean
            
            rmse = orig_std * torch.sqrt(torch.mean((y_test.flatten() - mean.flatten())**2))
            nlpd = -observed_pred.log_prob(y_test.flatten())
        
    return lml.item(), rmse.item(), nlpd.item()

## 1. Baseline: Exact Diffusion Kernel

In [ ]:
class ExactDiffusionKernel(gpytorch.kernels.Kernel):
    def __init__(self, adj, **kwargs):
        super().__init__(**kwargs)
        D_diag = adj.sum(dim=1)
        D_inv_sqrt = torch.diag(1.0 / torch.sqrt(D_diag + 1e-6))
        L = torch.eye(adj.size(0)).to(adj.device) - D_inv_sqrt @ adj @ D_inv_sqrt
        self.register_buffer("L", L)
        
        self.register_parameter(name="raw_beta", parameter=torch.nn.Parameter(torch.tensor(0.0)))
        self.register_parameter(name="raw_sigma", parameter=torch.nn.Parameter(torch.tensor(0.0)))
        
    @property
    def beta(self):
        return torch.nn.functional.softplus(self.raw_beta)
    
    @property
    def sigma(self):
        return torch.nn.functional.softplus(self.raw_sigma)

    def forward(self, x1, x2, diag=False, **params):
        K_full = self.sigma * torch.matrix_exp(-self.beta * self.L)
        i1, i2 = x1.long().flatten(), x2.long().flatten()
        if diag:
            return K_full[i1, i1]
        return K_full[i1][:, i2]

likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
baseline_kernel = ExactDiffusionKernel(adj_tensor).to(device)
model = GraphGP(X_train, Y_train, likelihood, baseline_kernel).to(device)

train_model(model, likelihood)
lml, rmse, nlpd = evaluate_model(model, likelihood, X_test, Y_test, orig_std)

print(f"Baseline (Exact Diffusion) - LML: {lml:.4f}, RMSE: {rmse:.4f}, NLPD: {nlpd:.4f}")
diffusion_rmse = rmse
diffusion_nlpd = nlpd

## 2. GRF Approximations

In [ ]:
walks_per_nodes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]
grf_diff_results = []
grf_gen_results = []

for walks in tqdm(walks_per_nodes, desc="Scaling Walkers"):
    sampler = GRFSampler(adj_tensor, walks_per_node=walks, p_halt=0.1, max_walk_length=MAX_WALK_LENGTH)
    rw_mats = [m.to(device) for m in sampler.sample_random_walk_matrices()]
    
    # 1. Diffusion-shaped GRF Kernel
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    kernel_diff = GRFDiffusionKernel(max_walk_length=MAX_WALK_LENGTH, rw_mats=rw_mats).to(device)
    model_diff = GraphGP(X_train, Y_train, likelihood, kernel_diff).to(device)
    train_model(model_diff, likelihood)
    res_diff = evaluate_model(model_diff, likelihood, X_test, Y_test, orig_std)
    grf_diff_results.append(res_diff)
    
    # 2. General (Arbitrary) GRF Kernel
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    kernel_gen = GRFGeneralKernel(max_walk_length=MAX_WALK_LENGTH, rw_mats=rw_mats).to(device)
    model_gen = GraphGP(X_train, Y_train, likelihood, kernel_gen).to(device)
    train_model(model_gen, likelihood)
    res_gen = evaluate_model(model_gen, likelihood, X_test, Y_test, orig_std)
    grf_gen_results.append(res_gen)

## 3. Visualization

In [ ]:
sns.set(style="whitegrid")
palette = sns.color_palette("colorblind", n_colors=4)

rmse_diff = [r[1] for r in grf_diff_results]
rmse_gen = [r[1] for r in grf_gen_results]
nlpd_diff = [r[2] for r in grf_diff_results]
nlpd_gen = [r[2] for r in grf_gen_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(walks_per_nodes, rmse_diff, 'o-', color=palette[0], label='GRF Diffusion-Shape')
ax1.plot(walks_per_nodes, rmse_gen, 's-', color=palette[1], label='GRF General (Arbitrary)')
ax1.axhline(y=diffusion_rmse, color='gray', linestyle='--', label='Exact Diffusion Baseline')
ax1.set_xscale('log')
ax1.set_xlabel('Walkers per Node')
ax1.set_ylabel('RMSE')
ax1.set_title('RMSE vs Number of Walkers')
ax1.legend()

ax2.plot(walks_per_nodes, nlpd_diff, 'o-', color=palette[0], label='GRF Diffusion-Shape')
ax2.plot(walks_per_nodes, nlpd_gen, 's-', color=palette[1], label='GRF General (Arbitrary)')
ax2.axhline(y=diffusion_nlpd, color='gray', linestyle='--', label='Exact Diffusion Baseline')
ax2.set_xscale('log')
ax2.set_xlabel('Walkers per Node')
ax2.set_ylabel('NLPD')
ax2.set_title('NLPD vs Number of Walkers')
ax2.legend()

plt.tight_layout()
plt.show()